In [1]:
# Install and import necessary libraries
!pip install torch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset

!pip install torchinfo
from torchinfo import summary

import numpy as np

from scipy.io import loadmat

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import LabelEncoder

import os

import pandas as pd

import random

import matplotlib.pyplot as plt

import seaborn as sns

In [2]:
# Configure training device (CPU by default; will use CUDA if available):
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Ensure reproducibility by fixing the random seed.
seed = 34

# Define function to handle reproducibility
def set_seed(seed=seed):
    # Fix seed for Python's built-in random module:
    random.seed(seed)
    
    # Fix seed for NumPy random operations:
    np.random.seed(seed)

    # Fix seed for PyTorch CPU operations:
    torch.manual_seed(seed)

    # Fix seed for GPU operations if CUDA is available
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Force deterministic behavior in CUDA convolution and backend operations:
    torch.backends.cudnn.deterministic = True

    # Disable CuDNN benchmarking to prevent selecting different algorithms per run
    torch.backends.cudnn.benchmark = False

# Ensure reproducibility:
set_seed(seed)

# Worker seed function for DataLoader:
def seed_worker(worker_id):

    # Ensure each worker is deterministic:
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

# Create a generator with fixed seed for shuffling:
g = torch.Generator()
g.manual_seed(seed)

# Print reproducibility message:
print("Reproducibility ensured.")

Using device: cpu
Reproducibility ensured.


/opt/conda/lib/python3.13/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [3]:
# Define relative paths of the FC matrices:
SC_FC_Coupling_Dir = "../../postprocessing/SC-FC_Coupling/network_SC_coupling.csv"

# Load dataset
df = pd.read_csv(SC_FC_Coupling_Dir )

# Create feature dataset X:
X = df.drop(columns=["Subject_ID", "Clinical_Group"])

# Create label dataset y
y = df["Clinical_Group"]

# Encode labels if categorical
le = LabelEncoder()
y = le.fit_transform(y)

# Convert lists to NumPy arrays:
X = np.array(X)
y = np.array(y)

# Print results:
first_dim, columns = X.shape
print(f"FC matrices shape: Amount = {first_dim}, Columns = {columns}")
print(f"Labels shape: Amount = {y.shape[0]}.")

FC matrices shape: Amount = 1083, Columns = 7
Labels shape: Amount = 1083.


In [4]:
# Add channel dimension for CNN input compatibility #
# CNN input shape: [Number of Subjects (N), Number of Channels (C), Matrix Rows or Height (H), Matrix Columns or Width (W) #
X = np.expand_dims(X, axis=1)

# Print new input dataset shape:
N, C, W = X.shape

print(f"CNN input shape: {X.shape}")
print(f"Number of FC Matrices (N): {N}")
print(f"Number of Channels (C): {C}")
print(f"Width / Columns (W): {W} ")

CNN input shape: (1083, 1, 7)
Number of FC Matrices (N): 1083
Number of Channels (C): 1
Width / Columns (W): 7 


In [5]:
# Perform Train / Validation / Test Data Splits #

# First split:
# 80%: Train + Validation.
# 20%: Test.

# Perform the Train + Validation and Test data split considering both data stratification (to preserve class proportion across splits) and random seed reproducibility:
X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=seed)

# Second split:
# From the remaining 80%:
# 75%: Train.
# 25%: Validation.

# Perform the Train and Validation data split considering both data stratification and random seed reproducibility:
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size=0.25, stratify=y_trainval, random_state=seed)

# Final proportions:
# 60% Train / 20% Validation / 20% Test

# Print final data split input shapes:
print("=== INPUT SHAPES ===")
print("Train input shape:", X_train.shape)
print("Validation input shape:", X_val.shape)
print("Test input shape:", X_test.shape)
print()

# Print final data split label shapes:
print("=== LABEL SHAPES ===")
print("Train label shape:", y_train.shape[0])
print("Validation label shape:", y_val.shape[0])
print("Test label shape:", y_test.shape[0])

=== INPUT SHAPES ===
Train input shape: (649, 1, 7)
Validation input shape: (217, 1, 7)
Test input shape: (217, 1, 7)

=== LABEL SHAPES ===
Train label shape: 649
Validation label shape: 217
Test label shape: 217


In [6]:
# Convert input data train, validation and test data splits into tensors:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

# Convert label train, validation and test splits into tensors:
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [7]:
print("Input data split tensor shapes:")
print("X_train: Number of ECG Signals (N) =", X_train_tensor.shape[0], ", Number of Channels (C)=", X_train_tensor.shape[1], ", ECG Signal Length (L) =", X_train_tensor.shape[2])
print("X_val: Number of ECG Signals (N) =", X_val_tensor.shape[0], ", Number of Channels (C)=", X_val_tensor.shape[1], ", ECG Signal Length (L) =", X_val_tensor.shape[2])
print("X_test: Number of ECG Signals (N) =", X_test_tensor.shape[0], ", Number of Channels (C)=", X_test_tensor.shape[1], ", ECG Signal Length (L) =", X_test_tensor.shape[2])

Input data split tensor shapes:
X_train: Number of ECG Signals (N) = 649 , Number of Channels (C)= 1 , ECG Signal Length (L) = 7
X_val: Number of ECG Signals (N) = 217 , Number of Channels (C)= 1 , ECG Signal Length (L) = 7
X_test: Number of ECG Signals (N) = 217 , Number of Channels (C)= 1 , ECG Signal Length (L) = 7


In [8]:
print("Label data split tensor shapes:")
print("y_train: Number of Labels (N) =", y_train_tensor.shape[0])
print("y_val:   Number of Labels (N) =", y_val_tensor.shape[0])
print("y_test:  Number of Labels (N) =", y_test_tensor.shape[0])

Label data split tensor shapes:
y_train: Number of Labels (N) = 649
y_val:   Number of Labels (N) = 217
y_test:  Number of Labels (N) = 217


In [9]:
# Define a flexible and configurable 1D CNN Model:

# Define a custom PyTorch CNN_1D class Neural Network (NN) model for 1D data.
# nn.Module is the base class for all neural networks in PyTorch.
class FlexibleCNN1D(nn.Module):

    # Define model architecture parameters:
    def __init__(self,

        # Input sequence length:
        input_length,

        # Number of output classes:
        num_classes=3,

        # Convolutional Architecture Parameters:
        conv_channels=[16, 32, 64], # Number of filters or feature vectors per convolutional layer.
        kernel_sizes=[3, 3, 3], # Kernel size for each convolutional layer.
        paddings=[1, 1, 1], # Padding for each convolutional layer.
        strides=[1, 1, 1], # Stride for each convolutional layer.

        activation="relu",  # Activation function type.

        pooling="max", # Pooling operation type: max or average.
        pool_sizes=[2, 2, 2], # Pool size per layer.

        # Fully Connected Architecture Parameters
        fc_layers=[128], # Number of neurons in each FC layer.

        dropout=0.3 # Dropout probability.
    ):

        # Call the parent class constructor from nn.Module:
        super().__init__()

        # Activation function selection:
        activation_dict = {"relu": nn.ReLU, "tanh": nn.Tanh, "sigmoid": nn.Sigmoid, "leakyrelu": nn.LeakyReLU, "elu": nn.ELU}
        act_fn = activation_dict[activation.lower()]

        # Pooling operation selection:
        if pooling.lower() == "max":
            pool_layer = nn.MaxPool1d
        elif pooling.lower() == "avg":
            pool_layer = nn.AvgPool1d
        else:
            raise ValueError("Pooling must be 'max' or 'avg'.")

        ### Dynamically Build Convolutional Layers ###
        
        # This section constructs the CNN feature extractor based on user-defined architecture parameters.

        # Initialize an empty list to store all convolutional layers sequentially:
        layers = []

        # Set initial number of input channels (1 for 1D ECG / time-series signals):
        in_channels = 1

        # Loop over each convolutional layer configuration.
        # Each zip iteration defines one Conv1D block (Conv → Batch Normalization → Activation → Pool)
        for out_channels, kernel, padding, stride, pool_size in zip(conv_channels, kernel_sizes, paddings, strides, pool_sizes):

            # Add current 1D convolution layer:
            layers.append(nn.Conv1d(in_channels=in_channels, out_channels=out_channels, kernel_size=kernel, padding=padding, stride=stride))

            # Add batch normalization:
            layers.append(nn.BatchNorm1d(out_channels))

            # Add activation function:
            layers.append(act_fn())

            # Add pooling layer:
            layers.append(pool_layer(kernel_size=pool_size))

            # Update input channels for the next convolutional layer:
            in_channels = out_channels

        # Combine all layers into a single sequential feature extractor:
        self.features = nn.Sequential(*layers)

        ### Automatically compute flattened size ###
        # This section determines the correct input size for the fully connected layers
        # by passing a dummy tensor through the convolutional feature extractor.

        # Disable gradient tracking:
        with torch.no_grad():

            # Create a dummy input tensor with (batch_size=1, channels=1, sequence_length=input_length):
            dummy = torch.zeros(1, 1, input_length)

            # Pass dummy input through the convolutional feature extractor:
            out = self.features(dummy)

            # Flatten the output from (batch, channels, length) into a single vector per sample:
            flattened_size = out.view(1, -1).shape[1]

        ### Dynamically Build Fully Connected Layers Layers ###

        # Initialize an empty list to store fully connected layers:
        classifier_layers = []

        # Set input size for the first fully connected layer as the length of the flattened output of the CNN:
        in_features = flattened_size

        # Loop through each fully connected layer configuration.
        # Each value in fc_layers represents the number of neurons in that layer.
        for neurons in fc_layers:

            # Add a fully connected (Linear) layer:
            classifier_layers.append(nn.Linear(in_features, neurons) )

            # Add activation function:
            classifier_layers.append(act_fn())

            # Add batch normalization:
            classifier_layers.append(nn.BatchNorm1d(neurons))

            # Add dropout for regularization:
            classifier_layers.append(nn.Dropout(dropout))

            # Update input size:
            in_features = neurons

        # Final output layer:
        classifier_layers.append(nn.Linear(in_features, num_classes))

        # Combine all fully connected layers into a single sequential module:
        self.classifier = nn.Sequential(*classifier_layers)

    ### Forward Pass ###
    def forward(self, x):

        # Pass input through convolutional feature extractor.
        # Output shape: (batch_size, channels, length).
        x = self.features(x)

        # Flatten feature maps into a 1D vector per sample.
        # Shape becomes: (batch_size, flattened_features).
        x = x.view(x.size(0), -1)

        # Pass flattened features through fully connected classifier:
        x = self.classifier(x)

        # Return final predictions (logits):
        return x

In [10]:
# Instantiate a model:
model = FlexibleCNN1D(y_train_tensor.shape[0])

# Show a summary of the model:
summary(model)

Layer (type:depth-idx)                   Param #
FlexibleCNN1D                            --
├─Sequential: 1-1                        --
│    └─Conv1d: 2-1                       64
│    └─BatchNorm1d: 2-2                  32
│    └─ReLU: 2-3                         --
│    └─MaxPool1d: 2-4                    --
│    └─Conv1d: 2-5                       1,568
│    └─BatchNorm1d: 2-6                  64
│    └─ReLU: 2-7                         --
│    └─MaxPool1d: 2-8                    --
│    └─Conv1d: 2-9                       6,208
│    └─BatchNorm1d: 2-10                 128
│    └─ReLU: 2-11                        --
│    └─MaxPool1d: 2-12                   --
├─Sequential: 1-2                        --
│    └─Linear: 2-13                      663,680
│    └─ReLU: 2-14                        --
│    └─BatchNorm1d: 2-15                 256
│    └─Dropout: 2-16                     --
│    └─Linear: 2-17                      387
Total params: 672,387
Trainable params: 672,387
Non-train

In [11]:
# Define a function for model training, validation and testing:
def train_model(model, train_loader, val_loader, test_loader, epochs, lr, criterion, device):

    # Define the optimizer to update model weights during training:
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # Training and Validation Loop #

    # Initialize an empty dictionary to store training metrics over epochs:
    history = {
        "train_acc": [], # Training accuracy per epoch.
        "val_acc": [], # Validation accuracy per epoch.
        "train_loss": [], # Training loss per epoch.
        "val_loss": [] # Validation loss per epoch.
    }

    # Loop over the dataset for a fixed number of epochs:
    for epoch in tqdm(range(epochs), desc="Epochs"):

        ### TRAINING PHASE ###
        
        # Set model to training mode:
        model.train()

        # Initialize training counters for accuracy computation:
        train_correct = 0 # Number of correct training predictions.
        train_total = 0 # Total number of training samples seen.

        # Initialize cumulative total training loss for the current epoch:
        epoch_t_loss = 0

        # Iterate over mini-batches from the training DataLoader:
        for t_inputs, t_labels in train_loader:

            # Move training input data and labels to the device:
            t_inputs, t_labels = t_inputs.to(device), t_labels.to(device)

            # Reset gradients from previous step:
            optimizer.zero_grad()

            # Perform forward pass to compute model predictions:
            t_outputs = model(t_inputs)

            # Compute loss between predictions and true labels:
            t_loss = criterion(t_outputs, t_labels)

            # Perform backward pass to compute gradients:
            t_loss.backward()

            # Update model parameters:
            optimizer.step()

            # Accumulate training batch loss:
            epoch_t_loss += t_loss.item()

            # Get predicted class labels:
            _, t_predicted = torch.max(t_outputs, 1)

            # Update total training sample count:
            train_total += t_labels.size(0)

            # Count correct training predictions:
            train_correct += (t_predicted == t_labels).sum().item()

        # Compute training accuracy for the epoch:
        epoch_train_acc = 100 * train_correct / train_total

        # Compute average training loss for the epoch:
        epoch_train_loss = epoch_t_loss / len(train_loader)

        # Store training accuracy of the current epoch:
        history["train_acc"].append(epoch_train_acc)

        # Store training loss of the current epoch:
        history["train_loss"].append(epoch_train_loss)

        ### VALIDATION PHASE ###

        # Set model to evaluation mode (disables dropout, batchnorm updates, etc.):
        model.eval()

        # Initialize validation counters for accuracy computation:
        val_correct = 0 # Number of correct validation predictions.
        val_total = 0 # Total number of validation samples seen.

        # Initialize cumulative total training loss for the current epoch:
        epoch_v_loss = 0

        # Disable gradient computation:
        with torch.no_grad():

            # Iterate over mini-batches from the validation DataLoader:
            for v_inputs, v_labels in val_loader:

                # Move validation input data and labels to the device:
                v_inputs, v_labels = v_inputs.to(device), v_labels.to(device)

                # Perform forward pass to compute model predictions (no gradient tracking):
                v_outputs = model(v_inputs)

                # Compute loss between predictions and true labels:
                v_loss = criterion(v_outputs, v_labels)
        
                # Accumulate validation batch loss:
                epoch_v_loss += v_loss.item()

                # Get predicted class labels:
                _, v_predicted = torch.max(v_outputs, 1)

                # Update total validation sample count:
                val_total += v_labels.size(0)

                # Count correct validation predictions:
                val_correct += (v_predicted == v_labels).sum().item()

        # Compute validation accuracy for this epoch:
        epoch_val_acc = 100 * val_correct / val_total

        # Compute average validation loss for the epoch:
        epoch_val_loss = epoch_v_loss / len(val_loader)
        
        # Store validation accuracy of the current epoch:
        history["val_acc"].append(epoch_val_acc)

        # Store validation loss of the current epoch:
        history["val_loss"].append(epoch_val_loss)


        # Print epoch summary:
        #print(
        #    f"Epoch {epoch+1}/{epochs} | "
        #    f"Training Loss: {epoch_train_loss:.4f} | "
        #    f"Validation Loss: {epoch_val_loss:.4f} | "
        #    f"Training Accuracy: {epoch_train_acc:.2f}% | "
        #    f"Validation Accuracy: {epoch_val_acc:.2f}%"
        #)

    # Helper function to plot training and validation accuracy:
    def plot_accuracies(history):
    
        # Create a new figure with the specified size:
        plt.figure(figsize=(8, 5))

        # Define Garnet & Ocean Blue color palette:
        granate = "#8B1A2B"
        granate_pastel = "#F2B6C1"
        
        ocean_blue = "#0077BE"
        ocean_blue_pastel = "#A7D3F2"
    
        # Plot training accuracy with circular markers:
        plt.plot(history['train_acc'], label='Train Accuracy', marker='o', color=granate_pastel, markerfacecolor=granate_pastel, markeredgecolor=granate, linewidth=2)
    
        # Plot validation accuracy with square markers:
        plt.plot(history['val_acc'], label='Validation Accuracy', marker='o', color=ocean_blue_pastel, markerfacecolor=ocean_blue_pastel, markeredgecolor=ocean_blue, linewidth=2)
    
        # Add a title to the graph:
        plt.title('Model Accuracy', fontsize=20)
    
        # Add x- and y-axis labels:
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy (%)')
    
        # Display the legend:
        plt.legend()
    
        # Add a dashed grid with slight transparency:
        plt.grid(True, linestyle='--', alpha=0.6)
    
        # Show the final plot:
        plt.show()

    # Helper function to plot training and validation losses:
    def plot_losses(history):

        # Define Garnet & Ocean Blue color palette:
        granate = "#8B1A2B"
        granate_pastel = "#F2B6C1"
        
        ocean_blue = "#0077BE"
        ocean_blue_pastel = "#A7D3F2"

        # Create a new figure with the specified size:
        plt.figure(figsize=(8, 5))
    
        # Plot training loss with circular markers:
        plt.plot(history['train_loss'], label='Train Loss', marker='o', color=granate_pastel, markerfacecolor=granate_pastel, markeredgecolor=granate, linewidth=2)
    
        # Plot validation accuracy with circular markers:
        plt.plot(history['val_loss'], label='Validation Loss', marker='o', color=ocean_blue_pastel, markerfacecolor=ocean_blue_pastel, markeredgecolor=ocean_blue, linewidth=2)

        # Add a title to the graph:
        plt.title('Model Loss', fontsize=20)

        # Add x- and y-axis labels:
        plt.xlabel('Epoch')
        plt.ylabel('Loss')

        # Display the legend:
        plt.legend()

        # Add a dashed grid with slight transparency:
        plt.grid(True, linestyle='--', alpha=0.6)

        # Show the final plot:
        plt.show()
    
    # Call the helper function using the history data to plot accuracies:
    plot_accuracies(history)

    # Call the helper function using the history data to plot losses:
    plot_losses(history)

    ### TESTING PHASE ###

    # Set model to evaluation mode (disables dropout, batchnorm updates, etc.):
    model.eval()
    
    # Initialize empty lists to store model predictions and true labels for evaluation:
    preds = []
    trues = []
    
    # Disable gradient computation:
    with torch.no_grad():
    
       # Iterate over mini-batches from the testing DataLoader:
        for ev_inputs, ev_labels in test_loader:
    
            # Move testing input data to the selected device:
            ev_inputs = ev_inputs.to(device)
    
            # Perform forward pass to compute model predictions (no gradient tracking):
            ev_outputs = model(ev_inputs)
    
            # Get predicted class labels:
            _, ev_predicted = torch.max(ev_outputs, 1)
    
            # Store predictions:
            preds.extend(ev_predicted.cpu().numpy())
    
            # Store true labels:
            trues.extend(ev_labels.numpy())
    
    # Compute overall test accuracy:
    acc = accuracy_score(trues, preds)

    # Print test accuracy:
    print("\nClassification Report:")
    print(classification_report(trues, preds))

    # Plot confusion matrix # 

    # Define label mapping:
    labels = ["ARR", "CHF", "NSR"]
    
    # Generate the confusion matrix using true and predicted labels:
    cm = confusion_matrix(trues, preds)

    # Convert confusion matrix to percentage:
    cm_percent = cm.astype('float') / cm.sum(axis=1, keepdims=True) * 100
    
    # Create a new figure with width 6 and height 5:
    plt.figure(figsize=(6, 5))
    
    # Display the confusion matrix as a heatmap:
    sns.heatmap(cm_percent, annot=True, fmt=".1f", cmap="Blues",  vmin=0, vmax=100, linewidths=0.8, linecolor="gray", xticklabels=labels, yticklabels=labels)
    
    # Add x- and y- axis labels:
    plt.xlabel("Predicted Label", fontweight='bold')
    plt.ylabel("True Label", fontweight='bold')
    
    # Add a title to the graph:
    plt.title("Confusion Matrix", fontsize=20)
    
    # Display the final plot:
    plt.show()

    return history

In [12]:
# Define initial model hyperparameters:
batch_size = 8
epochs = 50
lr = 1e-4

# Define the loss function:
criterion = nn.CrossEntropyLoss()

In [13]:
# Create TensorDatasets:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# Create DataLoaders (fully reproducible):
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, worker_init_fn=seed_worker, generator=g)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, worker_init_fn=seed_worker)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, worker_init_fn=seed_worker)

In [14]:
from torch.utils.data import TensorDataset, DataLoader

In [15]:
# Ensure reproducibility:
set_seed(seed)

# Instantiate a model object:
model = FlexibleCNN1D(
    input_length=X_train_tensor.shape[2]
).to(device)

# Train, validate and test an initial model:
history = FlexibleCNN1D(model=model, train_loader=train_loader, val_loader=val_loader, test_loader=test_loader, epochs=epochs, lr=lr, criterion=criterion, device=device)

ValueError: Expected more than 1 value per channel when training, got input size torch.Size([1, 64, 1])